<a href="https://colab.research.google.com/github/taiwoaquila/PE-ASD-Epigenetic-Convergence/blob/main/PE_ASD_Clean_Final_Results_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PE–ASD Integrative Epigenetic Study
## Clean final-results pipeline (reproducible consolidation and validation)

**Working title:** *Assessment of Epigenetic Convergence Between Preeclampsia-Associated Developmental Programming and Autism Spectrum Disorder*

### Scope
This notebook consolidates and audits the **final analyses already completed**. It does **not** rerun the original PE RRBS differential methylation, ASD buccal EWAS, or ASD brain differential methylation models. It does not overwrite source files.

It:
1. validates the final PE, ASD buccal, ASD brain, integration and enrichment outputs;
2. checks gene-symbol consistency and the six pre-specified three-way candidates;
3. summarises the nominal/FDR evidence without upgrading nominal findings;
4. audits pathway enrichment with within-gene-set and global Benjamini–Hochberg correction;
5. creates a clean final-results directory, report and file inventory.

**Important:** A formal overlap p-value is not calculated unless a complete, appropriate common testable-gene universe is supplied and verified. The 403 selected PE genes are not used as the background universe.

Run cells from top to bottom in Google Colab. The project directory must already be present in Google Drive.


In [ ]:
# CELL 1 — Mount Drive and define paths
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, re, json, shutil, warnings
import numpy as np
import pandas as pd
from datetime import datetime

warnings.filterwarnings("ignore")

PROJECT = Path("/content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026")
INTEGRATION = PROJECT / "integration"
ASD_DIR = PROJECT / "asd_analysis"
BRAIN_DIR = PROJECT / "asd_brain_analysis"
FINAL_DIR = INTEGRATION / "my final results"
OUT = FINAL_DIR / "Clean_Final_Pipeline"

TABLES = OUT / "Tables"
REPORTS = OUT / "Reports"
EVIDENCE = OUT / "Evidence"
for d in [TABLES, REPORTS, EVIDENCE]:
    d.mkdir(parents=True, exist_ok=True)

assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert INTEGRATION.exists(), f"Integration folder not found: {INTEGRATION}"

print("Project:", PROJECT)
print("Integration:", INTEGRATION)
print("Output:", OUT)
print("Run timestamp:", datetime.now().isoformat(timespec="seconds"))


Mounted at /content/drive
Project: /content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026
Integration: /content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026/integration
Output: /content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026/integration/my final results/Clean_Final_Pipeline
Run timestamp: 2026-11-03T04:56:50


In [ ]:
# CELL 2 — Load the final analysis outputs (no methylation models are rerun)

def require_file(path, label=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required {label or 'file'} not found: {path}")
    return path

def read_csv_required(path, label=None):
    path = require_file(path, label)
    df = pd.read_csv(path, low_memory=False)
    print(f"Loaded {label or path.name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
    return df

PE_FILE = INTEGRATION / "Shared_Early_Late_PE_genes.csv"
BUCCAL_GENE_FILE = INTEGRATION / "ASD_buccal_gene_level_Simes.csv"
INTEGRATED_FILE = INTEGRATION / "PE_ASD_integrated_gene_convergence.csv"
SIX_FILE = INTEGRATION / "Six_gene_complete_evidence_table.csv"
THREEWAY_FILE = INTEGRATION / "three_way_PE_buccal_brain_genes.csv"
ENRICHMENT_FILE = INTEGRATION / "PE_ASD_all_pathway_enrichment_results.csv"

pe = read_csv_required(PE_FILE, "shared early/late PE genes")
buccal_genes = read_csv_required(BUCCAL_GENE_FILE, "ASD buccal gene-level Simes")
integrated = read_csv_required(INTEGRATED_FILE, "integrated PE–ASD convergence")
six_evidence = read_csv_required(SIX_FILE, "six-gene evidence table")
threeway = read_csv_required(THREEWAY_FILE, "three-way overlap genes")
enrichment = read_csv_required(ENRICHMENT_FILE, "combined pathway enrichment")

# Confirm expected schema rather than silently guessing columns.
required_pe = ["SYMBOL_Early", "SYMBOL_Late", "direction_Early", "direction_Late"]
required_integrated = [
    "gene", "PE_shared", "ASD_buccal_nominal", "ASD_buccal_FDR",
    "ASD_prefrontal_cortex_nominal",
    "ASD_prefrontal_cortex_age_sex_nominal",
    "ASD_temporal_cortex_nominal",
    "ASD_temporal_cortex_age_sex_nominal",
    "ASD_cerebellum_nominal",
    "ASD_cerebellum_age_sex_nominal",
    "ASD_brain_any_region_nominal", "PE_buccal_brain_overlap"
]
required_enrichment = ["ID", "Description", "pvalue", "p.adjust", "geneID", "gene_set"]

for col in required_pe:
    if col not in pe.columns:
        raise ValueError(f"PE shared-gene file missing expected column: {col}")
for col in required_integrated:
    if col not in integrated.columns:
        raise ValueError(f"Integrated table missing expected column: {col}")
for col in required_enrichment:
    if col not in enrichment.columns:
        raise ValueError(f"Enrichment table missing expected column: {col}")

print("Core schemas verified.")


FileNotFoundError: Required shared early/late PE genes not found: /content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026/integration/Shared_Early_Late_PE_genes.csv

In [ ]:
from pathlib import Path

BASE = Path("/content/drive/MyDrive/Taiwo_PE_ASD_Integrative_study_2026")

for file in BASE.rglob("Shared_Early_Late_PE_genes.csv"):
    print("Found file:", file)

In [ ]:
# CELL 3 — Validate PE shared-gene set and gene symbols

pe["SYMBOL_Early"] = pe["SYMBOL_Early"].fillna("").astype(str).str.strip().str.upper()
pe["SYMBOL_Late"] = pe["SYMBOL_Late"].fillna("").astype(str).str.strip().str.upper()

symbol_consistency = pe["SYMBOL_Early"].eq(pe["SYMBOL_Late"])
pe_consistency_report = pd.DataFrame({
    "check": ["PE shared-gene rows", "Unique early symbols", "Unique late symbols",
              "Early/late symbol consistency"],
    "value": [
        len(pe),
        pe.loc[pe["SYMBOL_Early"].ne(""), "SYMBOL_Early"].nunique(),
        pe.loc[pe["SYMBOL_Late"].ne(""), "SYMBOL_Late"].nunique(),
        bool(symbol_consistency.all())
    ]
})
pe_consistency_report.to_csv(TABLES / "01_PE_gene_symbol_validation.csv", index=False)

if not symbol_consistency.all():
    mismatches = pe.loc[~symbol_consistency, ["SYMBOL_Early", "SYMBOL_Late"]]
    mismatches.to_csv(TABLES / "01b_PE_symbol_mismatches.csv", index=False)
    raise ValueError("Early/late PE gene-symbol mismatch detected. See 01b_PE_symbol_mismatches.csv")

pe_genes = set(pe.loc[pe["SYMBOL_Early"].ne(""), "SYMBOL_Early"])
print("PE shared genes:", len(pe_genes))
print("Early/late symbol consistency:", bool(symbol_consistency.all()))
display(pe_consistency_report)


In [ ]:
# CELL 4 — Validate integrated Boolean flags and reproduce overlap counts

boolean_cols = [c for c in required_integrated if c not in ["gene"]]
true_values = {"true", "1", "yes", "y"}

for c in boolean_cols:
    if integrated[c].dtype == bool:
        continue
    integrated[c] = integrated[c].astype(str).str.strip().str.lower().isin(true_values)

integrated["gene"] = integrated["gene"].fillna("").astype(str).str.strip().str.upper()

overlap_summary = pd.DataFrame([
    {"comparison": c, "number_of_genes": int(integrated[c].sum())}
    for c in boolean_cols
])
overlap_summary.to_csv(TABLES / "02_Verified_Overlap_Counts.csv", index=False)

threeway_rebuilt = integrated.loc[
    integrated["PE_buccal_brain_overlap"], ["gene"]
].drop_duplicates().sort_values("gene")

threeway_rebuilt.to_csv(TABLES / "03_Three_Way_Overlap_Candidates_Rebuilt.csv", index=False)

expected_candidates = {"ARID3A", "FAM120B", "QRFP", "RIN3", "SGTA", "SHH"}
observed_candidates = set(threeway_rebuilt["gene"])
print("Three-way candidate genes:", sorted(observed_candidates))

if observed_candidates != expected_candidates:
    print("WARNING: candidate list differs from the previously reported six-gene set.")
    print("Expected:", sorted(expected_candidates))
    print("Observed:", sorted(observed_candidates))
else:
    print("Six-gene candidate set matches the previously validated set.")

display(overlap_summary)


In [ ]:
# CELL 5 — Validate the six-gene evidence table

candidate_genes = ["ARID3A", "FAM120B", "QRFP", "RIN3", "SGTA", "SHH"]

# Find a gene-symbol column using the actual table contents.
gene_col = None
for c in ["gene", "SYMBOL", "Gene", "gene_symbol"]:
    if c in six_evidence.columns:
        gene_col = c
        break
if gene_col is None:
    for c in six_evidence.columns:
        vals = set(six_evidence[c].dropna().astype(str).str.upper())
        if len(vals.intersection(candidate_genes)) >= 4:
            gene_col = c
            break
if gene_col is None:
    raise ValueError("Could not identify the gene column in Six_gene_complete_evidence_table.csv")

six_evidence[gene_col] = six_evidence[gene_col].astype(str).str.strip().str.upper()
six_final = six_evidence[six_evidence[gene_col].isin(candidate_genes)].copy()

if set(six_final[gene_col]) != set(candidate_genes):
    raise ValueError("Six-gene evidence table does not contain exactly the expected six candidate genes.")

six_final = six_final.sort_values(gene_col)
six_final.to_csv(TABLES / "04_Six_Candidate_Complete_Evidence.csv", index=False)

# Compact candidate robustness summary, using recorded columns where present.
robust_cols = [
    gene_col, "PE_Early_direction", "PE_Late_direction",
    "PE_early_late_direction_relationship",
    "remains_candidate_age_sex_any_region",
    "remains_candidate_same_region_age_sex"
]
available_robust_cols = [c for c in robust_cols if c in six_final.columns]
if len(available_robust_cols) > 1:
    six_final[available_robust_cols].to_csv(
        TABLES / "05_Six_Candidate_Robustness_Summary.csv", index=False
    )

print("Six candidates retained:", len(six_final))
display(six_final[available_robust_cols] if len(available_robust_cols) > 1 else six_final)


In [ ]:
# CELL 6 — Audit the pathway enrichment results

from statsmodels.stats.multitest import multipletests

enrichment = enrichment.copy()
enrichment["pvalue"] = pd.to_numeric(enrichment["pvalue"], errors="coerce")
enrichment["p.adjust"] = pd.to_numeric(enrichment["p.adjust"], errors="coerce")

# Recalculate BH globally across every non-missing p-value in the combined table.
enrichment["global_BH_FDR"] = np.nan
ok = enrichment["pvalue"].notna()
if ok.any():
    enrichment.loc[ok, "global_BH_FDR"] = multipletests(
        enrichment.loc[ok, "pvalue"], method="fdr_bh"
    )[1]

# Recalculate BH within each gene set (matching the family definition stated here).
enrichment["within_gene_set_BH_FDR"] = np.nan
for gene_set, idx in enrichment.groupby("gene_set", dropna=False).groups.items():
    p = enrichment.loc[idx, "pvalue"]
    good = p.notna()
    if good.any():
        enrichment.loc[p.index[good], "within_gene_set_BH_FDR"] = multipletests(
            p.loc[good], method="fdr_bh"
        )[1]

enrichment.to_csv(TABLES / "06_Enrichment_FDR_Audit_All_Terms.csv", index=False)

enrichment_summary = (
    enrichment.groupby("gene_set", dropna=False)
    .agg(
        terms_tested=("pvalue", "count"),
        nominal_p05=("pvalue", lambda x: int((x < 0.05).sum())),
        original_adjusted_p05=("p.adjust", lambda x: int((x < 0.05).sum())),
        within_set_BH_p05=("within_gene_set_BH_FDR", lambda x: int((x < 0.05).sum())),
        global_BH_p05=("global_BH_FDR", lambda x: int((x < 0.05).sum())),
        minimum_p=("pvalue", "min"),
        minimum_original_adjusted_p=("p.adjust", "min"),
        minimum_global_BH=("global_BH_FDR", "min")
    ).reset_index()
)
enrichment_summary.to_csv(TABLES / "07_Enrichment_Summary_By_Gene_Set.csv", index=False)

prefrontal = enrichment[
    enrichment["gene_set"].astype(str).str.contains("prefrontal", case=False, na=False)
].copy()
prefrontal = prefrontal.sort_values("pvalue")
prefrontal.to_csv(TABLES / "08_Prefrontal_Enrichment_Audit.csv", index=False)

prefrontal_original_sig = prefrontal[
    pd.to_numeric(prefrontal["p.adjust"], errors="coerce") < 0.05
].copy()
prefrontal_original_sig.to_csv(
    TABLES / "09_Prefrontal_Original_Adjusted_Significant_Terms.csv", index=False
)

# Extract genes contributing to originally adjusted-significant prefrontal terms.
driver_rows = []
for _, row in prefrontal_original_sig.iterrows():
    genes = [
        g.strip() for g in re.split(r"[/,;|]", str(row.get("geneID", "")))
        if g.strip() and g.strip().lower() != "nan"
    ]
    for gene in genes:
        driver_rows.append({
            "pathway_id": row.get("ID"),
            "pathway": row.get("Description"),
            "gene_set": row.get("gene_set"),
            "ontology": row.get("ontology", ""),
            "gene": gene,
            "nominal_p": row.get("pvalue"),
            "original_adjusted_p": row.get("p.adjust"),
            "global_BH_FDR": row.get("global_BH_FDR"),
            "within_gene_set_BH_FDR": row.get("within_gene_set_BH_FDR")
        })

drivers = pd.DataFrame(driver_rows)
drivers.to_csv(TABLES / "10_Prefrontal_Pathway_Gene_Drivers.csv", index=False)

print("Enrichment rows:", len(enrichment))
print("Original adjusted-significant terms:", int((enrichment["p.adjust"] < 0.05).sum()))
print("Global BH-significant terms:", int((enrichment["global_BH_FDR"] < 0.05).sum()))
print("Prefrontal original adjusted-significant terms:", len(prefrontal_original_sig))
display(enrichment_summary)


In [ ]:
# CELL 7 — Audit the final source-result summaries and write statistical caveats

caveats = [
    "The PE shared-gene list contains 403 genes; verify the exact CpG-to-gene annotation and shared-gene definition in the original scripts.",
    "The PE RRBS cohort comprised 26 offspring: 9 controls, 9 early-onset PE and 8 late-onset PE.",
    "The original PE outputs reported 3,309 early-PE and 1,937 late-PE CpGs at q < 0.05; post-hoc coverage-filtered subsets were 775 and 84 sites, respectively.",
    "The post-hoc coverage screen does not automatically preserve the original FDR significance of the filtered subset.",
    "The ASD buccal analysis reported zero FDR-significant CpGs and zero FDR-significant gene-level Simes results.",
    "The ASD brain analysis reported zero FDR-significant CpGs and zero FDR-significant gene-level Simes results.",
    "The brain comparison was restricted to 267 PE-associated genes represented on the 450K array, not a genome-wide brain EWAS.",
    "The six three-way genes are exploratory candidates, not FDR-confirmed ASD genes or validated biomarkers.",
    "The three-way overlap count is descriptive. Do not report a formal overlap p-value without a verified common testable-gene universe.",
    "The six-gene candidate evidence includes mixed probe directions for some genes; do not imply uniform gene-wide methylation direction.",
    "The PE model outputs included convergence warnings; report model details and warnings transparently where relevant.",
    "The biological tissue represented by the PE offspring RRBS samples must be confirmed from the source study before naming it in the dissertation.",
    "Cross-tissue comparisons do not establish causality or identical methylation states across tissues.",
    "Prefrontal enrichment findings need interpretation in light of the small nominal overlap gene set and the multiple-testing family definition."
]
(REPORTS / "Statistical_and_Interpretive_Caveats.txt").write_text("\n".join("- " + x for x in caveats))

# Dataset and analysis summary based on validated project records.
dataset_summary = pd.DataFrame([
    ["PE offspring RRBS", "GSE245779", 26, "Control 9; Early_PE 9; Late_PE 8", "RRBS; hg38", "Completed; outputs already generated"],
    ["ASD buccal", "GSE50759", 96, "ASD 48; Control 48", "Illumina 450K; hg19 annotation", "Completed; outputs already generated"],
    ["ASD brain", "GSE53162", 40, "PFC 6 ASD/5 control; temporal 6/10; cerebellum 7/6", "Illumina 450K", "Completed; age and age+sex models"]
], columns=["dataset", "GEO_accession", "n_samples", "group_breakdown", "platform_or_build", "status"])
dataset_summary.to_csv(TABLES / "11_Dataset_Analysis_Summary.csv", index=False)

print("Caveat register and dataset summary saved.")
display(dataset_summary)


In [ ]:
# CELL 8 — Copy existing final figures/tables without moving or deleting source files

# Only copies files that exist. No wildcard-based claims that missing figures were generated.
figure_dir = FINAL_DIR / "Figures"
supp_dir = FINAL_DIR / "Supplementary Tables"
support_dir = FINAL_DIR / "Supporting Files"
for d in [figure_dir, supp_dir, support_dir]:
    d.mkdir(parents=True, exist_ok=True)

# Copy known existing Venn figure variants, if present.
figure_patterns = [
    "Venn_PE_ASD_all_brain_regions.*",
    "Venn_PE_ASD_prefrontal_cortex.*",
    "Venn_PE_ASD_temporal_cortex.*",
    "Venn_PE_ASD_cerebellum.*",
    "Venn_PE_ASD_age_sex_sensitivity.*"
]

copied = []
missing_patterns = []
for pattern in figure_patterns:
    matches = list(INTEGRATION.glob(pattern))
    if not matches:
        missing_patterns.append(pattern)
    for src in matches:
        dest = figure_dir / src.name
        if src.resolve() != dest.resolve():
            shutil.copy2(src, dest)
        copied.append(str(dest))

# Preserve key final tables in Supporting Files for a single handoff folder.
support_files = [
    PE_FILE, BUCCAL_GENE_FILE, INTEGRATED_FILE, SIX_FILE,
    THREEWAY_FILE, ENRICHMENT_FILE
]
for src in support_files:
    if src.exists():
        shutil.copy2(src, support_dir / src.name)

pd.DataFrame({"copied_file": copied}).to_csv(TABLES / "12_Copied_Figure_Inventory.csv", index=False)
(REPORTS / "Missing_Figure_Patterns.txt").write_text(
    "\n".join(missing_patterns) if missing_patterns else "All specified Venn figure patterns were found."
)

print("Existing figures copied:", len(copied))
print("Figure patterns not found:", missing_patterns)


In [ ]:
# CELL 9 — Generate consolidated report and file inventory

report_lines = [
    "PE–ASD INTEGRATIVE EPIGENETIC STUDY",
    "CLEAN FINAL-RESULTS PIPELINE REPORT",
    f"Generated: {datetime.now().isoformat(timespec='seconds')}",
    "",
    "PROJECT SCOPE",
    "This pipeline consolidated existing final outputs. It did not rerun original methylation models.",
    "",
    "DATASETS",
    dataset_summary.to_string(index=False),
    "",
    "PE SHARED-GENE VALIDATION",
    f"Rows in PE shared-gene file: {len(pe)}",
    f"Unique PE gene symbols: {len(pe_genes)}",
    f"Early/late symbol consistency: {bool(symbol_consistency.all())}",
    "",
    "OVERLAP COUNTS",
    overlap_summary.to_string(index=False),
    "",
    "THREE-WAY CANDIDATES",
    ", ".join(sorted(observed_candidates)),
    "",
    "ENRICHMENT AUDIT",
    f"Total enrichment rows: {len(enrichment)}",
    f"Original adjusted-p significant rows: {int((enrichment['p.adjust'] < 0.05).sum())}",
    f"Global BH FDR significant rows: {int((enrichment['global_BH_FDR'] < 0.05).sum())}",
    "",
    "ENRICHMENT BY GENE SET",
    enrichment_summary.to_string(index=False),
    "",
    "INTERPRETATION",
    "Nominal gene-level overlaps are exploratory and are not equivalent to FDR-significant ASD findings.",
    "No formal overlap significance is claimed without a verified common testable-gene universe.",
    "Cross-tissue overlap does not establish causality.",
    "",
    "CAVEATS",
    *["- " + x for x in caveats]
]
(REPORTS / "Consolidated_Final_Results_Report.txt").write_text("\n".join(report_lines))

inventory = []
for f in OUT.rglob("*"):
    if f.is_file():
        inventory.append({
            "relative_path": str(f.relative_to(OUT)),
            "size_bytes": f.stat().st_size
        })
pd.DataFrame(inventory).to_csv(OUT / "File_Inventory.csv", index=False)

print("Pipeline completed.")
print("Output folder:", OUT)
print("Report:", REPORTS / "Consolidated_Final_Results_Report.txt")
print("Inventory:", OUT / "File_Inventory.csv")


## Final interpretation guardrails

- Keep PE CpG-level FDR results distinct from the post-hoc coverage-screened subsets.
- Report ASD buccal and brain gene-level Simes findings as nominal/exploratory where appropriate.
- Do not label the six overlapping genes as FDR-significant or validated biomarkers.
- Do not claim formal overlap significance without a complete and appropriate common gene universe.
- Do not claim that cross-tissue overlap demonstrates causality.
- Do not rerun methylation models in this consolidation notebook.

The original analysis scripts and raw result files remain in their original project folders. This notebook creates a separate audit and consolidation directory.
